In [1]:
"""
Compute and store sentence embeddings for all Neo4j nodes.

What it does:
  1. Pulls every node's (name + docstring) from Neo4j
  2. Encodes with all-MiniLM-L6-v2  →  384-dim float vectors
  3. Writes them back as n.embedding on each node
  4. Prints a progress bar and final stats

"""

import logging
import time
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

import transformers
from transformers.utils import logging

In [2]:
import logging as py_logging

# Use the renamed standard logger for basicConfig
py_logging.basicConfig(
    level=py_logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = py_logging.getLogger(__name__)

# Config — change these to match your setup 
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "pytorch2026"         
MODEL_NAME     = "mixedbread-ai/mxbai-embed-large-v1"      
BATCH_SIZE     = 256                      # nodes encoded per batch
WRITE_BATCH    = 100                      # nodes written per Neo4j transaction


def fetch_all_nodes(session) -> list[dict]:
    """Pull id, name, docstring for every node in the graph."""
    result = session.run(
        """
        MATCH (n)
        RETURN elementId(n) AS eid,
               coalesce(n.qualified, n.name, '') AS name,
               coalesce(n.docstring, n.description, n.text, n.note, '') AS doc
        """
    )
    return [{"eid": r["eid"], "name": r["name"], "doc": r["doc"]}
            for r in result]


def build_text(record: dict) -> str:
    """Combine name + docstring into one input string for the encoder."""
    name = (record["name"] or "").strip()
    doc  = (record["doc"]  or "").strip()[:300]   # cap docstring at 300 chars
    return f"{name}. {doc}" if doc else name


def write_embeddings(session, batch: list[dict]) -> None:
    """
    Write embeddings back to Neo4j in one transaction.
    Uses elementId (Neo4j 5.x) for precise node matching.
    """
    session.run(
        """
        UNWIND $rows AS row
        MATCH (n) WHERE elementId(n) = row.eid
        SET n.embedding = row.embedding
        """,
        rows=batch,
    )


def main():
    log.info("Loading sentence encoder: %s", MODEL_NAME)
    encoder = SentenceTransformer(MODEL_NAME)

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

    try:
        # Fetch all nodes 
        log.info("Fetching nodes from Neo4j …")
        with driver.session() as session:
            nodes = fetch_all_nodes(session)
        log.info("Found %d nodes to embed.", len(nodes))

        if not nodes:
            log.warning("No nodes found. Is the DB populated?")
            return

        # Encode in batches 
        texts = [build_text(n) for n in nodes]
        log.info("Encoding %d texts in batches of %d …", len(texts), BATCH_SIZE)

        t0 = time.time()
        embeddings = encoder.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,   # unit-norm → cosine sim = dot product
        )
        elapsed = time.time() - t0
        log.info(
            "Encoding done in %.1fs  (%.0f nodes/sec)",
            elapsed, len(nodes) / elapsed,
        )

        # Write back to Neo4j 
        log.info("Writing embeddings to Neo4j (write batch = %d) …", WRITE_BATCH)

        write_rows = [
            {"eid": nodes[i]["eid"], "embedding": embeddings[i].tolist()}
            for i in range(len(nodes))
        ]

        with driver.session() as session:
            for start in range(0, len(write_rows), WRITE_BATCH):
                chunk = write_rows[start : start + WRITE_BATCH]
                session.execute_write(write_embeddings, chunk)
                if (start // WRITE_BATCH) % 20 == 0:
                    log.info(
                        "  Written %d / %d nodes …",
                        min(start + WRITE_BATCH, len(write_rows)),
                        len(write_rows),
                    )

        # Verify 
        with driver.session() as session:
            count = session.run(
                "MATCH (n) WHERE n.embedding IS NOT NULL RETURN count(n) AS c"
            ).single()["c"]

        log.info("━━━ Done ━━━")
        log.info("  Nodes with embeddings : %d / %d", count, len(nodes))
        log.info("  Embedding dimension   : %d", len(embeddings[0]))
        log.info("  Model                 : %s", MODEL_NAME)

    finally:
        driver.close()


if __name__ == "__main__":
    main()

23:42:25  INFO      Loading sentence encoder: mixedbread-ai/mxbai-embed-large-v1
23:42:25  INFO      Use pytorch device_name: cuda:0
23:42:25  INFO      Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1
23:42:26  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
23:42:26  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/modules.json "HTTP/1.1 200 OK"
23:42:27  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

23:42:27  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
23:42:28  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/config_sentence_transformers.json "HTTP/1.1 200 OK"
23:42:28  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

23:42:28  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
23:42:28  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/config_sentence_transformers.json "HTTP/1.1 200 OK"
23:42:29  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
23:42:29  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/README.md "HTTP/1.1 200 OK"
23:42:29  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/README.md "HTTP/1.1 200 OK"


...

23:43:18  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
23:43:18  INFO      HTTP Request: HEAD https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
23:43:19  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
23:43:19  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/models/mixedbread-ai/mxbai-embed-large-v1/b33106f585b9ce46904ad7443a3b52b7a63e231c/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

23:43:19  INFO      HTTP Request: GET https://huggingface.co/api/models/mixedbread-ai/mxbai-embed-large-v1 "HTTP/1.1 200 OK"
23:43:20  INFO      1 prompt is loaded, with the key: query
23:43:20  INFO      Fetching nodes from Neo4j …
23:43:20  INFO      Found 24485 nodes to embed.
23:43:20  INFO      Encoding 24485 texts in batches of 256 …


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

23:43:47  INFO      Encoding done in 27.4s  (895 nodes/sec)
23:43:47  INFO      Writing embeddings to Neo4j (write batch = 100) …
23:43:48  INFO        Written 100 / 24485 nodes …
23:43:50  INFO        Written 2100 / 24485 nodes …
23:43:52  INFO        Written 4100 / 24485 nodes …
23:43:53  INFO        Written 6100 / 24485 nodes …
23:43:55  INFO        Written 8100 / 24485 nodes …
23:43:57  INFO        Written 10100 / 24485 nodes …
23:43:58  INFO        Written 12100 / 24485 nodes …
23:44:00  INFO        Written 14100 / 24485 nodes …
23:44:02  INFO        Written 16100 / 24485 nodes …
23:44:03  INFO        Written 18100 / 24485 nodes …
23:44:05  INFO        Written 20100 / 24485 nodes …
23:44:07  INFO        Written 22100 / 24485 nodes …
23:44:08  INFO        Written 24100 / 24485 nodes …
23:44:09  INFO      ━━━ Done ━━━
23:44:09  INFO        Nodes with embeddings : 24485 / 24485
23:44:09  INFO        Embedding dimension   : 1024
23:44:09  INFO        Model                 : mixedbread